<a href="https://colab.research.google.com/github/4cekay/B101-Group2-NLP-Project/blob/main/DatasetCuration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Setup**

In [47]:
# will update later -- we can do the splitting, tokenizing, training config, etc. on another notebook

!pip install numpy torch datasets transformers evaluate --quiet

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/factory.py", line 169, in _make_candidate_from_dist
    base = self._installed_candidate_cache[dist.canonical_name]
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
KeyError: 'h11'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages

In [32]:
import pandas as pd
from datasets import load_dataset

# **Extraction**
[HF Dataset Processing](https://huggingface.co/docs/datasets/v1.4.0/processing.html)


Example: **Stanford Humanual-Email Dataset**

Columns:
- **completion:** ground-truth email reply
  - Main target text sample !!
- post_id (discard)
- user_id (discard)
- timestamp (discard)
- turn_id (discard)
  - Note: In conversation "turn 1" rows, I can take the original email content to use as a prompt for AI-generated samples to compare (w/ proper attribution)
- persona (discard)
- **prompt:** role and content
  - Will pull the initial email content of select conversations to use in prompting later.  
- metadata (discard)


In [3]:
# Load the raw dataset (https://huggingface.co/datasets/snap-stanford/humanual-email)

raw_dataset = load_dataset('snap-stanford/humanual-email')
raw_dataset


README.md:   0%|          | 0.00/2.71k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  675kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/val-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6377 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/536 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/130 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['completion', 'post_id', 'user_id', 'timestamp', 'turn_id', 'persona', 'prompt', 'metadata'],
        num_rows: 6377
    })
    test: Dataset({
        features: ['completion', 'post_id', 'user_id', 'timestamp', 'turn_id', 'persona', 'prompt', 'metadata'],
        num_rows: 536
    })
    val: Dataset({
        features: ['completion', 'post_id', 'user_id', 'timestamp', 'turn_id', 'persona', 'prompt', 'metadata'],
        num_rows: 130
    })
})

1. I'm gonna extract the initial email content of each conversation thread to use as prompts for our generative email samples later.

In [21]:
# extracting only the rows with turn_id == 1, indicating the first "turn" in a round of emails

initial_emails = raw_dataset.filter(lambda example: example["turn_id"] == 1)

In [22]:
# removing all unwanted features, keeping only the "completion" and "prompt"
initial_emails = initial_emails.remove_columns(["post_id", "user_id", "timestamp", "turn_id", "persona", "metadata"])

In [28]:
# extract the email content from each prompt,
# get rid of quotation marks from the beginning and end
# remove the old columns
email_prompts = initial_emails.map(
    lambda example: {
        "text": example["prompt"][0]["content"].strip('\'"') # get rid of quotes
    },
    remove_columns= ["completion", "prompt"]
)

Map:   0%|          | 0/3780 [00:00<?, ? examples/s]

Map:   0%|          | 0/371 [00:00<?, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

In [29]:
email_prompts

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 3780
    })
    test: Dataset({
        features: ['text'],
        num_rows: 371
    })
    val: Dataset({
        features: ['text'],
        num_rows: 96
    })
})

In [33]:
df_email_prompts = email_prompts["train"].to_pandas()

df_email_prompts

,text
0,Mike: Attached is our proposed amendment to t...
1,Mary - here is a draft of what the PGE interco...
2,"Last week, Eric Cope and I met with Erik Simps..."
3,Grant:\n\nI have incorporated the language you...
4,"with all the bad news, I hope you are ok."
...,...
3775,"On October 4th, I'll be trying to fill Rob Bra..."
3776,"Dear All,\n\n\tAs many of you may already know..."
3777,So far I have heard from two Customer Service ...
3778,Hi Tana.\n\nCould you please forward Ross or m...


2. Now, we take the actual email replies to use as training data.

In [44]:
email_replies = initial_emails.map(
    lambda example: {
        "sample_replies": example["completion"].strip('\'"')
    },
    remove_columns= ["prompt", "completion"]
)

Map:   0%|          | 0/3780 [00:00<?, ? examples/s]

Map:   0%|          | 0/371 [00:00<?, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

In [45]:
email_replies

DatasetDict({
    train: Dataset({
        features: ['sample_replies'],
        num_rows: 3780
    })
    test: Dataset({
        features: ['sample_replies'],
        num_rows: 371
    })
    val: Dataset({
        features: ['sample_replies'],
        num_rows: 96
    })
})

In [46]:
df_email_replies = email_replies["train"].to_pandas()

df_email_replies

,sample_replies
0,We are not interested in these changes to the ...
1,Since EPMI doesn't have any PGE interconnectio...
2,"Martin,\n\nDavid Pruner called about this meet..."
3,"Grant,\n I have reviewed your requested ..."
4,"Ian,\n\nThanks for your message. No change her..."
...,...
3775,"In the constant state of ""blur"" I'm in, I some..."
3776,"Dear Dr. Lay,\n\n\tMy name is Iris Mack. My f..."
3777,To add to the list the following have also bee...
3778,Do you have a full legal name for Equiva Trading?


In [ ]:
# csv export example (index=False to get rid of the index no. column)

df_email_replies.to_csv('email_samples.csv', index=False)

# **Compilation**

Work in progress--just pls keep track of all your sample tables 😛

and read thru the datasets docs i linked if u need anything, bc all of the functions r there